# Dense Index + Dense Cache Smoke Validation

목표:
1. 필요한 의존성만 즉시 설치
2. `dense_index/smoke`에 샘플 인덱스 저장
3. `build_dense_cache.py`가 BM25와 같은 캐시 형식(json per source + manifest)을 생성하는지 확인

In [ ]:
from pathlib import Path
import os
import importlib
import subprocess

repo = Path('/workspace/ragmed')
os.chdir(repo)
print('cwd:', Path.cwd())


def run(cmd: str):
    print('\n' + '=' * 100)
    print(cmd)
    print('=' * 100)
    completed = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    print(f'[exit_code] {completed.returncode}')
    return completed.returncode


def ensure_module(module_name: str, package_name: str | None = None):
    try:
        importlib.import_module(module_name)
        print(f'[ok] {module_name}')
    except ModuleNotFoundError:
        pkg = package_name or module_name
        print(f'[install] {module_name} -> {pkg}')
        run(f'pip install {pkg}')


# 필요한 것만 설치
ensure_module('langchain_text_splitters', 'langchain-text-splitters')
ensure_module('sentence_transformers', 'sentence-transformers')
ensure_module('faiss', 'faiss-cpu')

In [ ]:
# 1) data dense cache smoke
# - retrieval cache output: retrieval_cache/dense_smoke_data
# - index artifacts: dense_index/smoke/data
run(
    'python -B src/build_dense_cache.py '
    '--encoder BAAI/bge-m3 '
    '--top-k 5 '
    '--max-docs-per-folder 10 '
    '--queries-per-source 3 '
    '--output-root retrieval_cache/dense_smoke_data '
    '--index-root dense_index/smoke/data'
)

In [ ]:
# 2) KorMedMCQA dense cache smoke
# - retrieval cache output: retrieval_cache/dense_smoke_kormed
# - index artifacts: dense_index/smoke/kormedmcqa
run(
    'python -B src/build_kormedmcqa_dense_cache.py '
    '--encoder BAAI/bge-m3 '
    '--top-k 5 '
    '--max-docs-per-folder 10 '
    '--output-root retrieval_cache/dense_smoke_kormed '
    '--index-root dense_index/smoke/kormedmcqa'
)

In [ ]:
# 3) 결과 확인
# - dense_index/smoke 에 인덱스 파일 존재 여부
# - retrieval_cache/dense_smoke_* 에 source별 cache json 생성 여부
run('find dense_index/smoke -maxdepth 4 -type f | sort')
run('find retrieval_cache/dense_smoke_data -maxdepth 2 -type f | sort | head -n 80')
run('find retrieval_cache/dense_smoke_kormed -maxdepth 2 -type f | sort')